# Web Scraping Approach

A hybrid scraping approach was used where Selenium handled dynamic content loading via infinite scrolling, and BeautifulSoup was used to parse the fully loaded HTML. This ensured efficient extraction of structured data from dynamically rendered pages.

In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.firefox.service import Service
from webdriver_manager.firefox import GeckoDriverManager

from bs4 import BeautifulSoup
import pandas as pd
import time
import random

# ---------- SETUP ----------
options = webdriver.FirefoxOptions()
options.add_argument("--start-maximized")

driver = webdriver.Firefox(
    service=Service(GeckoDriverManager().install()),
    options=options
)

# ---------- CITY LIST ----------
cities = ["Chennai", "Bangalore", "Hyderabad", "Mumbai"]

all_data = []

# ---------- LOOP THROUGH CITIES ----------
for city in cities:
    print(f"\nScraping city: {city}")

    url = f"https://www.magicbricks.com/property-for-sale/residential-real-estate?bedroom=&proptype=Multistorey-Apartment,Builder-Floor-Apartment,Penthouse,Studio-Apartment,Residential-House,Villa&cityName={city}"

    driver.get(url)
    time.sleep(6)

    # ---------- INFINITE SCROLL ----------
    prev_count = 0
    same_count_rounds = 0

    while True:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(random.uniform(6, 8))

        cards = driver.find_elements(By.CLASS_NAME, "mb-srp__card")
        curr_count = len(cards)

        print(f"{city} -> Loaded: {curr_count}")

        if curr_count == prev_count:
            same_count_rounds += 1
        else:
            same_count_rounds = 0

        if same_count_rounds >= 5:
            break

        prev_count = curr_count

    print(f"{city} scrolling complete")

    # ---------- GET FULL HTML ----------
    soup = BeautifulSoup(driver.page_source, "html.parser")

    cards = soup.find_all("div", class_="mb-srp__card")

    # ---------- EXTRACT ----------
    for card in cards:

        def get_text(el):
            return el.get_text(strip=True) if el else None

        title = get_text(card.find("h2", class_="mb-srp__card--title"))

        price = get_text(card.find("div", class_="mb-srp__card__price--amount"))

        sqft_price = get_text(card.find("div", class_="mb-srp__card__price--size"))

        # ---------- PROPERTY NAME  ----------
        property_name = (
            get_text(card.find("a", class_="mb-srp__card__society--name")) or
            get_text(card.find("div", class_="mb-srp__card__developer--name--highlight")) or
            get_text(card.find("span", class_="mb-srp__card__developer--name--highlight"))
        )

        #---------- SUMMARY ----------- 
        summary_el = card.find("div", class_="mb-srp__card__summary")

        summary = summary_el.get_text(separator=" | ", strip=True) if summary_el else None


        posted_by = get_text(card.find("div", class_="mb-srp__card__ads--name"))

        # ---------- DESCRIPTION  ----------
        description = None

        # Case 1: <p> with <br><br>
        desc_p = card.find("p")
        if desc_p:
            description = desc_p.get_text(separator=" ", strip=True)

        # Case 2: fallback
        if not description:
            desc_div = card.find("div", class_="mb-srp__card--desc-lux--text")
            if desc_div:
                description = get_text(desc_div)

        all_data.append({
            "city": city,
            "title": title,
            "price": price,
            "sqft_price": sqft_price,
            "property_name": property_name,
            "summary": summary,
            "posted_by": posted_by,
            "description": description
        })

# ---------- CLOSE ----------
driver.quit()

# ---------- SAVE ----------
df = pd.DataFrame(all_data)

print(df.head())
print(df.info())
print("Total listings:", len(df))

df.to_csv("magicbricks_data_final_5.csv", index=False)


Scraping city: Chennai
Chennai -> Loaded: 60
Chennai -> Loaded: 90
Chennai -> Loaded: 120
Chennai -> Loaded: 150
Chennai -> Loaded: 180
Chennai -> Loaded: 210
Chennai -> Loaded: 240
Chennai -> Loaded: 270
Chennai -> Loaded: 300
Chennai -> Loaded: 330
Chennai -> Loaded: 360
Chennai -> Loaded: 390
Chennai -> Loaded: 420
Chennai -> Loaded: 450
Chennai -> Loaded: 480
Chennai -> Loaded: 510
Chennai -> Loaded: 540
Chennai -> Loaded: 570
Chennai -> Loaded: 600
Chennai -> Loaded: 630
Chennai -> Loaded: 660
Chennai -> Loaded: 690
Chennai -> Loaded: 720
Chennai -> Loaded: 750
Chennai -> Loaded: 780
Chennai -> Loaded: 810
Chennai -> Loaded: 840
Chennai -> Loaded: 870
Chennai -> Loaded: 900
Chennai -> Loaded: 930
Chennai -> Loaded: 960
Chennai -> Loaded: 990
Chennai -> Loaded: 1020
Chennai -> Loaded: 1050
Chennai -> Loaded: 1080
Chennai -> Loaded: 1110
Chennai -> Loaded: 1140
Chennai -> Loaded: 1170
Chennai -> Loaded: 1200
Chennai -> Loaded: 1230
Chennai -> Loaded: 1260
Chennai -> Loaded: 1290
Ch